# Databricks Demo: Lakehouse Essentials (SQL version)
Covers: namespaces, ingestion, Delta internals, CTAS, optimization, compute.



## 1. Introduction: the three-level namespace `catalog.schema.object`

In [0]:
CREATE CATALOG IF NOT EXISTS demo;
CREATE SCHEMA IF NOT EXISTS demo.sales;
CREATE VOLUME IF NOT EXISTS demo.sales.raw;
USE demo.sales;

In [0]:
SHOW CATALOGS;

catalog
claims
claims_dev
claims_lakebase
customers
ddev
demo
dev
fk_ce_2026_07_1372
sales_summary
samples


In [0]:
SHOW SCHEMAS IN demo;

databaseName
default
information_schema
sales


Fully qualified names look like `demo.sales.orders`. Open **Catalog Explorer** to show permissions, lineage and sample data.

## Generate sample data (pure SQL)
`INSERT OVERWRITE DIRECTORY` writes query results as files into the volume. Each batch goes to its own folder.

In [0]:
INSERT OVERWRITE DIRECTORY '/Volumes/demo/sales/raw/orders/batch1'
USING CSV OPTIONS (header 'true')
SELECT /*+ COALESCE(1) */
  id AS order_id,
  CAST(1 + pmod(hash(id, 'c'), 50) AS INT) AS customer_id,
  element_at(array('Laptop', 'Phone', 'Headphones', 'Monitor', 'Keyboard'), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) AS product,
  CAST(element_at(array(900, 600, 120, 250, 60), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) * (1 + pmod(hash(id, 'q'), 3)) AS INT) AS amount,
  element_at(array('NEW', 'PAID', 'SHIPPED'), CAST(1 + pmod(hash(id, 's'), 3) AS INT)) AS status,
  date_add(DATE'2026-01-01', CAST(pmod(hash(id, 'd'), 60) AS INT)) AS order_date
FROM range(1, 501) AS t(id);

In [0]:
LIST '/Volumes/demo/sales/raw/orders/batch1';

path,name,size,modification_time
/Volumes/demo/sales/raw/orders/batch1/_SUCCESS,_SUCCESS,0,1795543894000
/Volumes/demo/sales/raw/orders/batch1/_committed_8755418561239713965,_committed_8755418561239713965,113,1795543894000
/Volumes/demo/sales/raw/orders/batch1/_started_8755418561239713965,_started_8755418561239713965,0,1795543893000
/Volumes/demo/sales/raw/orders/batch1/part-00000-tid-8755418561239713965-88574dda-8fc3-4549-a5ef-4d1687db93c4-171-1-c000.csv,part-00000-tid-8755418561239713965-88574dda-8fc3-4549-a5ef-4d1687db93c4-171-1-c000.csv,17873,1795543893000


## 2. Data ingestion methods

### 2a. UI upload
In the sidebar choose **+ New > Add or upload data** to upload a CSV into a table (show this live).
### 2b. `read_files` (query files directly)

In [0]:
SELECT * FROM read_files('/Volumes/demo/sales/raw/orders/*/*.csv', format => 'csv', header => true) LIMIT 10;

order_id,customer_id,product,amount,status,order_date,_rescued_data
1,10,Phone,1200,SHIPPED,2026-02-16,null
2,17,Laptop,1800,NEW,2026-01-14,null
3,14,Laptop,900,NEW,2026-01-07,null
4,18,Headphones,240,NEW,2026-01-29,null
5,48,Phone,1200,NEW,2026-02-11,null
6,36,Headphones,360,PAID,2026-01-19,null
7,9,Monitor,750,NEW,2026-01-22,null
8,12,Phone,1200,NEW,2026-02-21,null
9,40,Keyboard,120,SHIPPED,2026-01-13,null
10,16,Headphones,120,PAID,2026-03-01,null


### `read_files` for unstructured files (images)
`read_files` also reads **binary** files with `format => 'binaryFile'`. Instead of inferring columns, every file becomes **one row** with a fixed schema: `path`, `modificationTime`, `length` and `content` (the raw bytes as BINARY). This works for any file type: images, PDFs, audio.

Combine it with AI functions to actually *understand* the content:
* `ai_query` — send the bytes to a vision model (describe, classify, extract)
* `ai_parse_document` — extract structured fields from images and PDFs

**Videos:** `binaryFile` reads a video file's raw bytes, but no built-in function processes video directly. Extract frames into image files first, or use `ai_transcribe` for the audio track.

### 2c. Load files into a Delta table (CTAS over `read_files`)
`CREATE TABLE AS SELECT` on top of `read_files` loads all the files into a managed Delta table in one statement.

In [0]:
CREATE OR REPLACE TABLE demo.sales.orders_bronze AS
SELECT * FROM read_files('/Volumes/demo/sales/raw/orders/*/*.csv', format => 'csv', header => true);

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) AS rows_loaded FROM demo.sales.orders_bronze;

rows_loaded
500


### 2d. Append new files with `INSERT INTO ... SELECT`
A second batch of files arrives. We point `read_files` at only the new folder and append.

In [0]:
INSERT OVERWRITE DIRECTORY '/Volumes/demo/sales/raw/orders/batch2'
USING CSV OPTIONS (header 'true')
SELECT /*+ COALESCE(1) */
  id AS order_id,
  CAST(1 + pmod(hash(id, 'c'), 50) AS INT) AS customer_id,
  element_at(array('Laptop', 'Phone', 'Headphones', 'Monitor', 'Keyboard'), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) AS product,
  CAST(element_at(array(900, 600, 120, 250, 60), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) * (1 + pmod(hash(id, 'q'), 3)) AS INT) AS amount,
  element_at(array('NEW', 'PAID', 'SHIPPED'), CAST(1 + pmod(hash(id, 's'), 3) AS INT)) AS status,
  date_add(DATE'2026-01-01', CAST(pmod(hash(id, 'd'), 60) AS INT)) AS order_date
FROM range(501, 701) AS t(id);

In [0]:
INSERT INTO demo.sales.orders_bronze
SELECT * FROM read_files('/Volumes/demo/sales/raw/orders/batch2/*.csv', format => 'csv', header => true);

num_affected_rows,num_inserted_rows
200,200


In [0]:
SELECT COUNT(*) AS rows_loaded FROM demo.sales.orders_bronze;

rows_loaded
700


### 2e. Ingest multiple file formats (JSON, Parquet, ORC, Avro)
The same `read_files` function handles many file formats. We write the same sample orders in four different formats, then load them all into a single Delta table. If you omit `format =>`, `read_files` auto-detects the format from the file extension.

In [0]:
-- 2e write JSON batch
INSERT OVERWRITE DIRECTORY '/Volumes/demo/sales/raw/orders/formats/json'
USING JSON
SELECT /*+ COALESCE(1) */
  id AS order_id,
  CAST(1 + pmod(hash(id, 'c'), 50) AS INT) AS customer_id,
  element_at(array('Laptop', 'Phone', 'Headphones', 'Monitor', 'Keyboard'), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) AS product,
  CAST(element_at(array(900, 600, 120, 250, 60), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) * (1 + pmod(hash(id, 'q'), 3)) AS INT) AS amount,
  element_at(array('NEW', 'PAID', 'SHIPPED'), CAST(1 + pmod(hash(id, 's'), 3) AS INT)) AS status,
  date_add(DATE'2026-01-01', CAST(pmod(hash(id, 'd'), 60) AS INT)) AS order_date
FROM range(701, 801) AS t(id);

In [0]:
-- 2e write Parquet batch
INSERT OVERWRITE DIRECTORY '/Volumes/demo/sales/raw/orders/formats/parquet'
USING PARQUET
SELECT /*+ COALESCE(1) */
  id AS order_id,
  CAST(1 + pmod(hash(id, 'c'), 50) AS INT) AS customer_id,
  element_at(array('Laptop', 'Phone', 'Headphones', 'Monitor', 'Keyboard'), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) AS product,
  CAST(element_at(array(900, 600, 120, 250, 60), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) * (1 + pmod(hash(id, 'q'), 3)) AS INT) AS amount,
  element_at(array('NEW', 'PAID', 'SHIPPED'), CAST(1 + pmod(hash(id, 's'), 3) AS INT)) AS status,
  date_add(DATE'2026-01-01', CAST(pmod(hash(id, 'd'), 60) AS INT)) AS order_date
FROM range(801, 901) AS t(id);

In [0]:
-- 2e write ORC batch
INSERT OVERWRITE DIRECTORY '/Volumes/demo/sales/raw/orders/formats/orc'
USING ORC
SELECT /*+ COALESCE(1) */
  id AS order_id,
  CAST(1 + pmod(hash(id, 'c'), 50) AS INT) AS customer_id,
  element_at(array('Laptop', 'Phone', 'Headphones', 'Monitor', 'Keyboard'), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) AS product,
  CAST(element_at(array(900, 600, 120, 250, 60), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) * (1 + pmod(hash(id, 'q'), 3)) AS INT) AS amount,
  element_at(array('NEW', 'PAID', 'SHIPPED'), CAST(1 + pmod(hash(id, 's'), 3) AS INT)) AS status,
  date_add(DATE'2026-01-01', CAST(pmod(hash(id, 'd'), 60) AS INT)) AS order_date
FROM range(901, 1001) AS t(id);

In [0]:
-- 2e write Avro batch
INSERT OVERWRITE DIRECTORY '/Volumes/demo/sales/raw/orders/formats/avro'
USING AVRO
SELECT /*+ COALESCE(1) */
  id AS order_id,
  CAST(1 + pmod(hash(id, 'c'), 50) AS INT) AS customer_id,
  element_at(array('Laptop', 'Phone', 'Headphones', 'Monitor', 'Keyboard'), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) AS product,
  CAST(element_at(array(900, 600, 120, 250, 60), CAST(1 + pmod(hash(id, 'p'), 5) AS INT)) * (1 + pmod(hash(id, 'q'), 3)) AS INT) AS amount,
  element_at(array('NEW', 'PAID', 'SHIPPED'), CAST(1 + pmod(hash(id, 's'), 3) AS INT)) AS status,
  date_add(DATE'2026-01-01', CAST(pmod(hash(id, 'd'), 60) AS INT)) AS order_date
FROM range(1001, 1101) AS t(id);

In [0]:
-- 2e list files per format
LIST '/Volumes/demo/sales/raw/orders/formats/json';
LIST '/Volumes/demo/sales/raw/orders/formats/parquet';
LIST '/Volumes/demo/sales/raw/orders/formats/orc';
LIST '/Volumes/demo/sales/raw/orders/formats/avro';

path,name,size,modification_time
/Volumes/demo/sales/raw/orders/formats/json/_SUCCESS,_SUCCESS,0,1795543906000
/Volumes/demo/sales/raw/orders/formats/json/_committed_2849606234931114562,_committed_2849606234931114562,114,1795543906000
/Volumes/demo/sales/raw/orders/formats/json/_started_2849606234931114562,_started_2849606234931114562,0,1795543905000
/Volumes/demo/sales/raw/orders/formats/json/part-00000-tid-2849606234931114562-6d7fbf30-5176-4610-b54e-3e6c99931467-181-1-c000.json,part-00000-tid-2849606234931114562-6d7fbf30-5176-4610-b54e-3e6c99931467-181-1-c000.json,11017,1795543905000


path,name,size,modification_time
/Volumes/demo/sales/raw/orders/formats/parquet/_SUCCESS,_SUCCESS,0,1795543907000
/Volumes/demo/sales/raw/orders/formats/parquet/_committed_3000004585494157151,_committed_3000004585494157151,124,1795543907000
/Volumes/demo/sales/raw/orders/formats/parquet/_started_3000004585494157151,_started_3000004585494157151,0,1795543906000
/Volumes/demo/sales/raw/orders/formats/parquet/part-00000-tid-3000004585494157151-8a480a94-a248-4e25-abdc-2a855bafc9be-182-1.c000.snappy.parquet,part-00000-tid-3000004585494157151-8a480a94-a248-4e25-abdc-2a855bafc9be-182-1.c000.snappy.parquet,3201,1795543906000


path,name,size,modification_time
/Volumes/demo/sales/raw/orders/formats/orc/_SUCCESS,_SUCCESS,0,1795543908000
/Volumes/demo/sales/raw/orders/formats/orc/_committed_862273939929788179,_committed_862273939929788179,119,1795543908000
/Volumes/demo/sales/raw/orders/formats/orc/_started_862273939929788179,_started_862273939929788179,0,1795543907000
/Volumes/demo/sales/raw/orders/formats/orc/part-00000-tid-862273939929788179-89f42329-c11c-4f83-be96-5ad2d4e3445d-183-1-c000.snappy.orc,part-00000-tid-862273939929788179-89f42329-c11c-4f83-be96-5ad2d4e3445d-183-1-c000.snappy.orc,1556,1795543908000


path,name,size,modification_time
/Volumes/demo/sales/raw/orders/formats/avro/_SUCCESS,_SUCCESS,0,1795543910000
/Volumes/demo/sales/raw/orders/formats/avro/_committed_3496225096313543591,_committed_3496225096313543591,121,1795543910000
/Volumes/demo/sales/raw/orders/formats/avro/_started_3496225096313543591,_started_3496225096313543591,0,1795543909000
/Volumes/demo/sales/raw/orders/formats/avro/part-00000-tid-3496225096313543591-fdff0d3d-b9a6-490c-9443-4e638440e767-184-1-c000.snappy.avro,part-00000-tid-3496225096313543591-fdff0d3d-b9a6-490c-9443-4e638440e767-184-1-c000.snappy.avro,1851,1795543910000


In [0]:
-- 2e preview with auto-detection
-- No format => needed: read_files infers it from the file extension
SELECT * FROM read_files('/Volumes/demo/sales/raw/orders/formats/json') LIMIT 5;
SELECT * FROM read_files('/Volumes/demo/sales/raw/orders/formats/parquet') LIMIT 5;

amount,customer_id,order_date,order_id,product,status,_rescued_data
1800,26,2026-01-12,701,Laptop,SHIPPED,null
250,4,2026-01-10,702,Monitor,PAID,null
1800,12,2026-01-29,703,Phone,PAID,null
750,33,2026-01-21,704,Monitor,SHIPPED,null
60,6,2026-02-12,705,Keyboard,SHIPPED,null


order_id,customer_id,product,amount,status,order_date,_rescued_data
801,35,Monitor,750,SHIPPED,2026-02-26,null
802,5,Keyboard,60,SHIPPED,2026-02-09,null
803,46,Phone,600,NEW,2026-03-01,null
804,34,Keyboard,120,SHIPPED,2026-02-19,null
805,12,Monitor,750,NEW,2026-01-10,null


In [0]:
-- 2e load all formats into one table
-- One Delta table fed by four different file formats
CREATE OR REPLACE TABLE demo.sales.orders_formats AS
SELECT 'json' AS source_format, order_id, customer_id, product, amount, status, CAST(order_date AS DATE) AS order_date
FROM read_files('/Volumes/demo/sales/raw/orders/formats/json')
UNION ALL
SELECT 'parquet', order_id, customer_id, product, amount, status, CAST(order_date AS DATE) AS order_date
FROM read_files('/Volumes/demo/sales/raw/orders/formats/parquet')
UNION ALL
SELECT 'orc', order_id, customer_id, product, amount, status, CAST(order_date AS DATE) AS order_date
FROM read_files('/Volumes/demo/sales/raw/orders/formats/orc')
UNION ALL
SELECT 'avro', order_id, customer_id, product, amount, status, CAST(order_date AS DATE) AS order_date
FROM read_files('/Volumes/demo/sales/raw/orders/formats/avro');

num_affected_rows,num_inserted_rows


In [0]:
-- 2e verify rows per format
SELECT source_format, COUNT(*) AS rows_loaded
FROM demo.sales.orders_formats
GROUP BY source_format
ORDER BY source_format;

source_format,rows_loaded
avro,100
json,100
orc,100
parquet,100


In [0]:
SELECT * FROM demo.sales.orders_formats LIMIT 5;

source_format,order_id,customer_id,product,amount,status,order_date
json,701,26,Laptop,1800,SHIPPED,2026-01-12
json,702,4,Monitor,250,PAID,2026-01-10
json,703,12,Phone,1800,PAID,2026-01-29
json,704,33,Monitor,750,SHIPPED,2026-01-21
json,705,6,Keyboard,60,SHIPPED,2026-02-12


Expect 700 rows. Here you choose which files to read. Running the insert again would add the same rows twice. Managed connectors (next sections) track what was already loaded for you.

### 2f. Connect to cloud storage (S3, ADLS or GCS)
You need your own bucket or container for this. The menu labels can differ slightly between releases.

**One-time setup (admin or a user with `CREATE EXTERNAL LOCATION`)**
1. **Create a storage credential.** Open **Catalog**, then **Add** and **Add a storage credential**. Use an IAM role ARN on AWS, an access connector or service principal on Azure, or a service account on GCP.
2. **Create an external location.** Open **Catalog > External Data > External Locations > Create location**. Enter a name, the storage URL (for example `s3://my-bucket/landing/`) and pick the credential. Click **Test connection**, then save.
3. **Grant access.** Open the external location, then **Permissions**, and grant `READ FILES` (and `WRITE FILES` if needed) to your users or group.

**Use it**
4. Query the files directly with `read_files`, then load them with CTAS (same pattern as 2b and 2c). Edit the commented cells below with your path.

In [0]:
-- Optional: create the external location in SQL instead of the UI
-- CREATE EXTERNAL LOCATION IF NOT EXISTS `demo_landing`
--   URL 's3://my-bucket/landing/'
--   WITH (STORAGE CREDENTIAL `my_storage_credential`);
-- GRANT READ FILES ON EXTERNAL LOCATION `demo_landing` TO `account users`;

In [0]:
-- SELECT * FROM read_files('s3://my-bucket/landing/', format => 'csv', header => true) LIMIT 10;
-- CREATE OR REPLACE TABLE demo.sales.cloud_orders AS
-- SELECT * FROM read_files('s3://my-bucket/landing/', format => 'csv', header => true);

![](/Workspace/Users/deepaksinghal@cyntexa.com/Demo/unity_catalog_s3_external_location_1m28s-5m50s_1794992717455.gif)

### 2g. Connect to a SaaS app with Lakeflow Connect (for example Salesforce)
This is a UI walkthrough, with no code.

**Before you start**
* A Salesforce user that Databricks can use, with API access and access to the objects you want.
* `CREATE CONNECTION` on the metastore (admins have it) to create a new connection, or `USE CONNECTION` on an existing one.
* Privileges to create tables in the destination catalog and schema.

**Steps**
1. In the sidebar, click **Data Ingestion**.
2. On the **Add data** page, under **Databricks connectors**, click **Salesforce**. The ingestion wizard opens.
3. **Connection:** pick an existing connection, or click **Create connection**. Sign in to Salesforce and approve access. On a first-time sign-in, Salesforce may need an admin to approve the Databricks connected app.
4. **Pipeline:** enter a unique pipeline name and choose the destination catalog and schema (for example `demo.sales`).
5. **Source:** choose the Salesforce objects to ingest, or all tables in the source schema (up to 250 objects per pipeline). Optionally set column selection and history tracking.
6. **Schedule:** add a schedule (for example every hour), then click **Save and run pipeline**.
7. **Check the result:** when the run finishes, open Catalog Explorer, go to the destination schema and query a table. Table names are lowercase versions of the Salesforce object names.

**Other SaaS and database sources.** Use the same flow for the other tiles under **Databricks connectors**, such as Salesforce Marketing Cloud, Google Analytics 4, Workday, ServiceNow or SQL Server. Available connectors depend on your workspace and release. Each connector has its own authentication step (OAuth or credentials) when you create the connection.

**No data movement option.** Lakehouse Federation queries an external source in place instead of copying it.

Create the silver orders table used in the rest of the demo:

![databricks_salesforce_ingest_walkthrough.gif](./databricks_salesforce_ingest_walkthrough.gif "databricks_salesforce_ingest_walkthrough.gif")

In [0]:
CREATE OR REPLACE TABLE demo.sales.orders AS
SELECT order_id, customer_id, product, amount, status, CAST(order_date AS DATE) AS order_date
FROM demo.sales.orders_bronze;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM demo.sales.orders LIMIT 5;

order_id,customer_id,product,amount,status,order_date
1,10,Phone,1200,SHIPPED,2026-02-16
2,17,Laptop,1800,NEW,2026-01-14
3,14,Laptop,900,NEW,2026-01-07
4,18,Headphones,240,NEW,2026-01-29
5,48,Phone,1200,NEW,2026-02-11


## 3. Delta Lake internals

### How Delta stores a table in cloud storage
A Delta table is just a directory in cloud storage (S3, ADLS or GCS) holding two kinds of things: **immutable Parquet data files** and a **`_delta_log`** folder. The log, not the data files, is the source of truth: a table at version N is defined by replaying the log up to commit N.

**Directory layout**
```
<table root>/
├── part-00000-...snappy.parquet     <- data files (immutable, written once)
└── _delta_log/
    ├── 00000000000000000000.json          <- commit 0 (transaction log entry)
    ├── 00000000000000000000.json.crc      <- checksum sidecar for commit 0
    ├── 00000000000000000001.json          <- commit 1
    ├── 00000000000000000010.checkpoint.parquet   <- checkpoint every 10 commits
    └── _last_checkpoint                    <- tiny JSON pointer to newest checkpoint
```

**1. Commit files (`N.json`) — the transaction log**
One small JSON file per table version, named by zero-padded version number. A commit is atomic: the file is written with put-if-absent semantics, so a version either fully exists or does not exist at all — that is what makes Delta ACID on cheap object storage. Each file lists **actions**, and each action type contains:
* `add` — a new data file now part of the table: path, size, partition values, and **per-file stats** (min/max/null counts per column) that later queries use for data skipping without reading the file
* `remove` — a data file made obsolete by this commit, with a deletion timestamp. The Parquet file is not deleted yet, only unlinked — that is what makes time travel possible until `VACUUM` runs
* `metaData` — the table definition: schema, partition columns, table id, table properties
* `protocol` — minimum reader/writer feature versions a client must support
* `commitInfo` — who/what/when: operation (WRITE, UPDATE, MERGE, ...), timestamps, user, and operation metrics (rows written, files touched)

**2. Checkpoint files (`.checkpoint.parquet`)**
Replaying thousands of JSON commits would get slow, so every 10 commits (`delta.checkpointInterval`) Delta writes a Parquet checkpoint that summarizes the **entire current log state** — the full list of live `add` files, `metaData` and `protocol` — up to that version. To open a table, a reader loads the latest checkpoint plus only the few JSON commits after it. `_last_checkpoint` is a tiny JSON file pointing at the newest checkpoint (version, file count).

**3. CRC files (`.crc`)**
Every commit and checkpoint file has a sidecar `.crc` file holding a **CRC32 checksum** of that log file (plus its length). Delta uses these during log retention and cleanup to detect **corrupted or truncated log files** — for example a half-written commit left behind by a failed cloud upload. It is integrity checking for the log itself, not validation of your table data. If a checksum does not match, Delta knows the log entry is untrustworthy.

**Why this design matters**
* **ACID on object storage:** concurrent writers commit by atomically creating the next `N.json`; losers of a race retry (optimistic concurrency control).
* **Time travel:** `VERSION AS OF N` simply replays the log to commit N; the older data files are still in cloud storage until `VACUUM`.
* **Fast reads:** per-file min/max stats in `add` actions let queries skip whole files (data skipping) and stream readers process only new commits.

In [0]:
DESCRIBE DETAIL demo.sales.orders;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,dbe020ed-5b85-4eae-9621-55c7f44316d0,demo.sales.orders,null,,2026-11-24T18:12:03.543Z,2026-11-24T18:12:05.000Z,List(),List(),1,4395,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.checkpoint.writeStatsAsStruct -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.checkpoint.writeStatsAsJson -> false, io.unitycatalog.tableId -> c1129750-72d5-4a8b-b0c9-1f14c7038d26, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-72baa661-0b8a-447c-95ea-07d2052aac0a, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-cf610c29-3cd0-435c-90bf-32b13d2be9a6, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
-- 3. describe / describe extended variants
DESCRIBE demo.sales.orders;

col_name,data_type,comment
order_id,int,null
customer_id,int,null
product,string,null
amount,int,null
status,string,null
order_date,date,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
DESCRIBE EXTENDED demo.sales.orders;

col_name,data_type,comment
order_id,int,null
customer_id,int,null
product,string,null
amount,int,null
status,string,null
order_date,date,null
# Clustering Information,,
# col_name,data_type,comment
,,
# Detailed Table Information,,


In [0]:
UPDATE demo.sales.orders SET status = 'SHIPPED' WHERE order_id = 1;

num_affected_rows
1


In [0]:
DELETE FROM demo.sales.orders WHERE order_id = 2;

num_affected_rows
1


In [0]:
DESCRIBE HISTORY demo.sales.orders;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-24T18:12:10.695Z,71730268075694,deepaksinghal@cyntexa.com,DELETE,"Map(predicate -> [""(order_id#16594 = 2)""])","Map(jobName -> null, jobOwnerId -> null, workspaceId -> 7474644987270626, jobId -> , jobRunId -> , runId -> , triggerType -> null)",null,01f1e832-73c0-1800-941f-7f805b621db0,null,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 516, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 336, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 180)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
1,2026-11-24T18:12:09.326Z,71730268075694,deepaksinghal@cyntexa.com,UPDATE,"Map(predicate -> [""(order_id#16043 = 1)""])","Map(jobName -> null, jobOwnerId -> null, workspaceId -> 7474644987270626, jobId -> , jobRunId -> , runId -> , triggerType -> null)",null,01f1e832-73c0-17ff-b50a-a73effd111c7,null,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2026, numDeletionVectorsUpdated -> 0, scanTimeMs -> 962, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 2731, rewriteTimeMs -> 1049)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13
0,2026-11-24T18:12:04.673Z,71730268075694,deepaksinghal@cyntexa.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.checkpoint.writeStatsAsStruct"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.checkpoint.writeStatsAsJson"":""false"",""io.unitycatalog.tableId"":""c1129750-72d5-4a8b-b0c9-1f14c7038d26"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-72baa661-0b8a-447c-95ea-07d2052aac0a"",""delta.enableInCommitTimestamps"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-cf610c29-3cd0-435c-90bf-32b13d2be9a6""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)","Map(jobName -> null, jobOwnerId -> null, workspaceId -> 7474644987270626, jobId -> , jobRunId -> , runId -> , triggerType -> null)",null,01f1e832-73c0-17fd-9a70-d818a6ffc975,null,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 700, numOutputBytes -> 4395)",null,Databricks-Runtime/19.15.x-aarch64-photon-scala2.13


Time travel and restore. On a fresh run, version 0 is the original table, 1 is after the UPDATE and 2 is after the DELETE. Check the history above if yours differs.

In [0]:
SELECT * FROM demo.sales.orders VERSION AS OF 1 WHERE order_id IN (1, 2);

order_id,customer_id,product,amount,status,order_date
2,17,Laptop,1800,NEW,2026-01-14
1,10,Phone,1200,SHIPPED,2026-02-16


In [0]:
RESTORE TABLE demo.sales.orders TO VERSION AS OF 1;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
7126,2,1,1,4395,4395


### Look inside the table
The `_metadata` column shows which Parquet files hold the rows. These are the data files that the `_delta_log` commits add and remove.

In [0]:
SELECT _metadata.file_path, _metadata.file_size, COUNT(*) AS rows_in_file
FROM demo.sales.orders
GROUP BY _metadata.file_path, _metadata.file_size;

file_path,file_size,rows_in_file
s3://dbstorage-prod-kpkgo/uc/7beac3db-3351-459d-a166-d9578b2b6b54/d269a4d8-23e1-4cb0-8439-b52b00e6fc5e/__unitystorage/catalogs/e8496579-33ec-4542-9645-196fc1857e85/tables/c1129750-72d5-4a8b-b0c9-1f14c7038d26/part-00000-78436f6a-a551-43d2-9f3f-99b58bf596ac.c000.zstd.parquet,4395,699
s3://dbstorage-prod-kpkgo/uc/7beac3db-3351-459d-a166-d9578b2b6b54/d269a4d8-23e1-4cb0-8439-b52b00e6fc5e/__unitystorage/catalogs/e8496579-33ec-4542-9645-196fc1857e85/tables/c1129750-72d5-4a8b-b0c9-1f14c7038d26/part-00000-956d0530-2f4b-4abf-84dd-b913e4c1a4e4.c000.zstd.parquet,2731,1


`VACUUM` deletes unreferenced files older than the retention period (7 days by default), which limits time travel.

In [0]:
VACUUM demo.sales.orders DRY RUN;

path


## 4. CTAS: CREATE TABLE AS SELECT

In [0]:
CREATE OR REPLACE TABLE demo.sales.daily_revenue
COMMENT 'Gold: daily revenue'
AS
SELECT order_date, SUM(amount) AS revenue, COUNT(*) AS orders
FROM demo.sales.orders
GROUP BY order_date;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM demo.sales.daily_revenue ORDER BY order_date LIMIT 10;

order_date,revenue,orders
2026-01-01,14190,15
2026-01-02,8360,9
2026-01-03,9530,11
2026-01-04,5070,6
2026-01-05,13100,16
2026-01-06,11900,16
2026-01-07,6850,10
2026-01-08,4910,10
2026-01-09,12810,11
2026-01-10,10730,16


`CREATE OR REPLACE` is atomic and keeps table history. Compare with a clone:

In [0]:
CREATE OR REPLACE TABLE demo.sales.orders_clone SHALLOW CLONE demo.sales.orders;

source_table_size,source_num_of_files,num_of_synced_transactions,num_removed_files,num_copied_files,removed_files_size,copied_files_size
7126,2,null,0,0,0,0


In [0]:
SELECT COUNT(*) AS rows_in_clone FROM demo.sales.orders_clone;

rows_in_clone
700


## 5. Optimization

Create a table with many small files. The `REPARTITION` hint forces many output files. Optimized writes on your workspace may merge some of them, so check the file count.

In [0]:
CREATE OR REPLACE TABLE demo.sales.orders_small AS
SELECT /*+ REPARTITION(40) */ * FROM demo.sales.orders;

num_affected_rows,num_inserted_rows


In [0]:
INSERT INTO demo.sales.orders_small SELECT /*+ REPARTITION(40) */ * FROM demo.sales.orders;

num_affected_rows,num_inserted_rows
700,700


In [0]:
DESCRIBE DETAIL demo.sales.orders_small;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,e4468b30-d354-4982-8253-d5505c8ca13e,demo.sales.orders_small,null,,2026-11-24T18:12:34.011Z,2026-11-26T09:31:04.000Z,List(),List(),41,99813,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.checkpoint.writeStatsAsStruct -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.checkpoint.writeStatsAsJson -> false, io.unitycatalog.tableId -> 6d1008e0-808a-4288-a5c5-d59e8c104f07, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-03e26beb-f773-42a7-b41b-adeea22d3af5, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-8ced1233-e280-47c0-916d-f2363e8ed70c, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
OPTIMIZE demo.sales.orders_small;

path,metrics
,"Map(recompressionCodec -> null, partitionsOptimized -> 0, numTableColumnsWithStats -> 6, skippedArchivedFiles -> 0, clusteringStats -> null, totalFilesSkipped -> 0, totalClusterParallelism -> 8, numFilesSkippedToReduceWriteAmplification -> 0, totalScheduledTasks -> 1, endTimeMs -> 1795685467875, startTimeMs -> 1795685466012, filesAdded -> Map(totalSize -> 7869, avg -> 7869.0, min -> 7869, totalFiles -> 1, max -> 7869), numBatches -> 1, numBins -> 0, deletionVectorStats -> Map(numDeletionVectorsRemoved -> 0, numDeletionVectorRowsRemoved -> 0), filesRemoved -> Map(totalSize -> 99813, avg -> 2434.4634146341464, min -> 2281, totalFiles -> 41, max -> 7746), totalTaskExecutionTimeMs -> 225, zOrderStats -> null, preserveInsertionOrder -> true, totalConsideredFiles -> 41, numBytesSkippedToReduceWriteAmplification -> 0, clusteringMetrics -> null, numTableColumns -> 6, numFilesSortedWithinFiles -> 0, numFilesAdded -> 1, numFilesRemoved -> 41, fileAgeHistogram -> null, autoCompactParallelismStats -> null)"


In [0]:
DESCRIBE DETAIL demo.sales.orders_small;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,e4468b30-d354-4982-8253-d5505c8ca13e,demo.sales.orders_small,null,,2026-11-24T18:12:34.011Z,2026-11-26T09:31:08.000Z,List(),List(),1,7928,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.checkpoint.writeStatsAsStruct -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.checkpoint.writeStatsAsJson -> false, io.unitycatalog.tableId -> 6d1008e0-808a-4288-a5c5-d59e8c104f07, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-03e26beb-f773-42a7-b41b-adeea22d3af5, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-8ced1233-e280-47c0-916d-f2363e8ed70c, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


`numFiles` drops after `OPTIMIZE`. Now Liquid Clustering for data skipping:

In [0]:
OPTIMIZE demo.sales.orders_clustered;

path,metrics
,"Map(recompressionCodec -> null, partitionsOptimized -> 0, numTableColumnsWithStats -> 6, skippedArchivedFiles -> 0, clusteringStats -> null, totalFilesSkipped -> 0, totalClusterParallelism -> 8, numFilesSkippedToReduceWriteAmplification -> 0, totalScheduledTasks -> 0, endTimeMs -> 1795685512680, startTimeMs -> 1795685509547, filesAdded -> Map(totalSize -> 0, avg -> 0.0, min -> null, totalFiles -> 0, max -> null), numBatches -> 0, numBins -> 0, deletionVectorStats -> Map(numDeletionVectorsRemoved -> 0, numDeletionVectorRowsRemoved -> 0), filesRemoved -> Map(totalSize -> 0, avg -> 0.0, min -> null, totalFiles -> 0, max -> null), totalTaskExecutionTimeMs -> 0, zOrderStats -> null, preserveInsertionOrder -> false, totalConsideredFiles -> 1, numBytesSkippedToReduceWriteAmplification -> 0, clusteringMetrics -> Map(numFilesSkippedAfterExpansion -> 0, coordinatorEventMetrics -> null, totalLogicalSizeOfDataToRewriteInBytes -> 0, logicalSizeOfFilesClassifiedToIntermediateNodesInBytes -> 0, numCompactionTasksPlannedUndoneDueToPO -> 0, compressionRatioFromSampling -> -1.0, numLeafNodesCompacted -> 0, nodeMinNumFilesToCompact -> 4, totalSizeOfSmallFilesSkippedAfterExpansionInBytes -> 0, pipelinedAssemblerMetrics -> null, compressionRatioSamplingTimeMs -> 0, predicates -> null, isPOTriggered -> false, numSamplingJobs -> 0, clusterThresholdStrategy -> log, compactionType -> null, sizeOfTableInBytesBeforeLazyClustering -> 4469, numFilesClassifiedToLeafNodes -> 1, timeMetrics -> Map(metadataCreationTimeMs -> 1189, totalGetFilesForNodeCallsTimeMs -> 0, optimizerTimeMs -> 53, classifierTimeMs -> 130, totalSamplingTimeMs -> 0, metadataLoadTimeMs -> 0, parentIndexBuildTimeMs -> 0), approxClusteringQuality -> null, numClusteringTasksNotPlannedDueToPO -> 0, numThreadsForClassifier -> null, totalSizeOfFilesSkippedAfterExpansionInBytes -> 0, splitStrategy -> defaultSplitStrategy, maxOptimizeBatchesInParallel -> 2, compressionRatioChosen -> -1.0, maxIterations -> 5, numLeafNodesExpanded -> 0, numIntermediateNodesCompactedUndoneDueToPO -> 0, numIdealFilesWithTrimmedStringMaxValue -> 0, numCompactionTasksPlanned -> 0, numFilesClassifiedToIntermediateNodes -> 0, numNullValuesEncounteredForTransforms -> 0, numLeafNodesClustered -> 0, compressionRatioNumSamples -> 0, numSmallFilesSkippedAfterExpansion -> 0, compressionRatioStalenessReason -> n/a, minFileSize -> 16777216, pipelinedTotalBatchWaitTimeMs -> 0, numLeafNodesCompactedUndoneDueToPO -> 0, numIdealFiles -> 0, currentIteration -> 1, numClusteringTasksPlanned -> 0, numGetFilesForNodeCalls -> 0, approxClusteringCoverage -> null, isNewMetadataCreated -> true, compressionRatioTriggeredScaling -> false, numIntermediateNodesCompacted -> 0, pipelinedPostFirstBatchWaitTimeMs -> 0, numFilesSkippedByPredicates -> 0, withPredicates -> false, isFull -> false, totalLogicalSizeOfDataToCompactInBytes -> 0, totalLogicalSizeOfDataToCompactInBytesUndoneDueToPO -> 0, totalSizeOfDataToRewriteInBytes -> 0, compressionRatioFromPersistedEntry -> false, numDowncastDimensions -> null, totalSizeOfDataToCompactInBytes -> 0, numIntermediateNodesClustered -> 0, maxFileSize -> 67108864, compressionRatioSource -> n/a, totalSizeOfDataToClusterInBytesNotPlannedDueToPO -> 0, clusteringStrategy -> sizeAware, pipelinedTotalPlanningTimeMs -> 0, numOptimizeBatchesPlanned -> 0, totalLogicalSizeOfFilesSkippedAfterExpansionInBytes -> 0, numFilesSkippedWithoutStats -> 0, isPipelinedOptimizer -> false, sizeOfFilesClassifiedToIntermediateNodesInBytes -> 0, sizeOfFilesClassifiedToLeafNodesInBytes -> 4469, pipelinedOptimizerStatus -> null, totalLogicalSizeOfSmallFilesSkippedAfterExpansionInBytes -> 0, approxClusteringQualityPerColumn -> null, numAddedFilesWithSameMinMaxOnClusteringColumns -> null, logicalSizeOfFilesClassifiedToLeafNodesInBytes -> 4469, totalSizeOfDataToCompactInBytesUndoneDueToPO -> 0), numTableColumns -> 6, numFilesSortedWithinFiles -> 0, numFilesAdded -> 0, numFilesRemoved -> 0, fileAgeHistogram -> null, autoComp

In [0]:
describe detail demo.sales.orders_clustered

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,9ad96c48-f243-4c29-9415-51d588589740,demo.sales.orders_clustered,null,,2026-11-26T09:31:46.424Z,2026-11-26T09:31:55.000Z,List(),"List(customer_id, order_date)",1,4469,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.checkpoint.writeStatsAsStruct -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.checkpoint.writeStatsAsJson -> false, io.unitycatalog.tableId -> 922ba965-5e05-4df6-9810-7a3402965ba7, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-0f8f3e56-857e-4983-b65b-8f5dff3290bb, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-17894ae1-a6fe-427d-938d-0e8618f35f8a, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
-- 5. describe table / show create table (clustering keys)
DESCRIBE TABLE demo.sales.orders_clustered;

col_name,data_type,comment
order_id,int,null
customer_id,int,null
product,string,null
amount,int,null
status,string,null
order_date,date,null
# Clustering Information,,
# col_name,data_type,comment
customer_id,int,null
order_date,date,null


In [0]:
SHOW CREATE TABLE demo.sales.orders_clustered;

createtab_stmt
"CREATE TABLE demo.sales.orders_clustered ( order_id INT, customer_id INT, product STRING COLLATE UTF8_BINARY, amount INT, status STRING COLLATE UTF8_BINARY, order_date DATE) USING delta DEFAULT COLLATION UTF8_BINARY CLUSTER BY (customer_id,order_date) TBLPROPERTIES ( 'delta.checkpoint.writeStatsAsJson' = 'false', 'delta.checkpoint.writeStatsAsStruct' = 'true', 'delta.checkpointPolicy' = 'v2', 'delta.enableDeletionVectors' = 'true', 'delta.enableInCommitTimestamps' = 'true', 'delta.enableRowTracking' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.inCommitTimestamp' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.feature.rowTracking' = 'supported', 'delta.feature.v2Checkpoint' = 'supported', 'delta.parquet.compression.codec' = 'zstd', 'delta.parquet.format.version' = '2.12.0', 'delta.parquet.format.version.afe.internal' = '2.12.0')"


In [0]:
SELECT COUNT(*) AS orders, SUM(amount) AS revenue
FROM demo.sales.orders_small
WHERE customer_id = 7;

orders,revenue
11,10360


In [0]:
CREATE OR REPLACE TABLE demo.sales.orders_clustered CLUSTER BY (customer_id, order_date) AS
SELECT * FROM demo.sales.orders_small;

num_affected_rows,num_inserted_rows


In [0]:
SELECT COUNT(*) AS orders, SUM(amount) AS revenue
FROM demo.sales.orders_clustered
WHERE customer_id = 7;

orders,revenue
11,10360


## 6. Compute

| Type | Use for |
|---|---|
| All-purpose cluster | Interactive notebooks and development |
| Job compute | Scheduled production workflows, cheaper and ephemeral |
| SQL warehouse (Classic, Pro, Serverless) | SQL, BI and dashboards |
| Serverless compute | Notebooks, jobs and pipelines with no cluster management |

**Live walkthrough**
1. Open **Compute > Create compute** and show runtime version, node type, autoscaling, auto-termination, Photon and spot instances.
2. Show access modes (standard vs dedicated) and cluster policies.
3. Open **SQL Warehouses** and show size, scaling and auto-stop.

Check what this notebook is running on:![db-guide_compute.webp](./db-guide_compute.webp "db-guide_compute.webp")![simple-form.png](./simple-form.png "simple-form.png")

In [0]:
SELECT current_version() AS runtime, current_user() AS user_name, current_catalog() AS catalog_name;

runtime,user_name,catalog_name
"Map(dbr_version -> null, dbsql_version -> 2026.46, u_build_hash -> 977caddbb1f5c46019706353401a32b8cbbd0466, r_build_hash -> f4d566620bd900e1856140b34f3a97cd992e1aac)",deepaksinghal@cyntexa.com,demo


In [0]:
SELECT
  CASE
    WHEN current_version().dbr_version IS NOT NULL THEN 'Classic cluster (DBR ' || current_version().dbr_version || ')'
    WHEN current_version().dbsql_version IS NOT NULL THEN 'Serverless / SQL warehouse'
    ELSE 'Unknown'
  END AS compute_type,
  current_version().dbsql_version AS version;

compute_type,version
Serverless / SQL warehouse,2026.46


In [0]:
SELECT compute, statement_text, start_time
FROM system.query.history
ORDER BY start_time DESC
LIMIT 5;

compute,statement_text,start_time
"Map(type -> WAREHOUSE, cluster_id -> null, warehouse_id -> c5bf9dda5ab27d2f)",SELECT MAX(rejection_rate) AS highest_plan_rejection_rate FROM claims.analytics.weekly_rejection_rate WHERE week_start = DATE'2026-06-22',2026-11-24T08:00:08.718Z
"Map(type -> WAREHOUSE, cluster_id -> null, warehouse_id -> c5bf9dda5ab27d2f)","BEGIN CREATE OR REPLACE TEMPORARY VIEW `temp_mv__01f1cc5af8ce1965bac87558ab29eddf` WITH METRICS LANGUAGE YAML AS $$ version: '1.1' source: tpca.gold.fact_bookings dimensions: - name: booking_date expr: booking_date comment: Date the booking was made - name: cabin_class expr: cabin_class comment: Cabin class for the flight segment - name: passenger_id expr: passenger_id comment: Passenger FK (join key to dim_passenger) - name: passenger_nation_key expr: passenger_nation_key comment: Passenger nation FK (join key to dim_geography) - name: route_id expr: route_id comment: Route FK (join key to dim_route) - name: airline_id expr: airline_id comment: Airline FK (join key to dim_airline) measures: - name: net_revenue expr: round(SUM(net_revenue),2) display_name: Net Revenue format: type: currency currency_code: USD decimal_places: type: exact places: 2 - name: net_revenue_millions expr: round(MEASURE(net_revenue) / 1000000, 2) display_name: Net Revenue ($M) comment: Net revenue expressed in millions of USD format: type: currency currency_code: USD decimal_places: type: exact places: 2 - name: gross_revenue expr: SUM(gross_revenue) display_name: Gross Revenue format: type: currency currency_code: USD decimal_places: type: exact places: 2 - name: booking_count expr: COUNT(DISTINCT booking_id) display_name: Distinct Bookings - name: passenger_count expr: COUNT(DISTINCT passenger_id) display_name: Active Passengers - name: avg_discount expr: AVG(discount_rate) display_name: Average Discount format: type: percentage decimal_places: type: exact places: 2 - name: avg_flight_duration expr: AVG(DATEDIFF(arrival_date, departure_date)) display_name: Average Flight Duration (days) - name: avg_booking_value expr: MEASURE(net_revenue) / MEASURE(booking_count) display_name: Average Booking Value format: type: currency currency_code: USD decimal_places: type: exact places: 2 $$ ; SELECT `cabin_class` `Fact_Bookings__cabin_class`,MEASURE(`avg_flight_duration`) `measure(Fact_Bookings__avg_flight_duration)` FROM temp_mv__01f1cc5af8ce1965bac87558ab29eddf GROUP BY `Fact_Bookings__cabin_class`; END;",2026-11-23T21:30:50.362Z
"Map(type -> WAREHOUSE, cluster_id -> null, warehouse_id -> c5bf9dda5ab27d2f)","BEGIN CREATE OR REPLACE TEMPORARY VIEW `temp_mv__01f1cc5af87512d885e12bdfbebc7ed1` WITH METRICS LANGUAGE YAML AS $$ version: '1.1' source: tpca.gold.dim_airline dimensions: - name: airline_id expr: airline_id comment: Airline primary key - name: airline_name expr: airline_name comment: Airline name - name: revenue_tier expr: ""CASE WHEN airline_name IN ('Eastern Pacific', 'AmeriJet', 'Gulf Express', 'Nordic Air', 'Sahara Airlines') THEN 'Top 5 Airlines' ELSE 'Rest of Network' END"" comment: Top 5 airlines by revenue vs rest (based on current data) $$ ; CREATE OR REPLACE TEMPORARY VIEW `temp_mv__01f1cc5af8ce1965bac87558ab29eddf` WITH METRICS LANGUAGE YAML AS $$ version: '1.1' source: tpca.gold.fact_bookings dimensions: - name: booking_date expr: booking_date comment: Date the booking was made - name: cabin_class expr: cabin_class comment: Cabin class for the flight segment - name: passenger_id expr: passenger_id comment: Passenger FK (join key to dim_passenger) - name: passenger_nation_key expr: passenger_nation_key comment: Passenger nation FK (join key to dim_geography) - name: route_id expr: route_id comment: Route FK (join key to dim_route) - name: airline_id expr: airline_id comment: Airline FK (join key to dim_airline) measures: - name: net_revenue expr: round(SUM(net_revenue),2) display_name: Net Revenue format: type: currency currency_code: USD decimal_places: type: exact places: 2 - name: net_reven

### Compute price comparison (approximate list prices)

Databricks measures computing power in **DBUs** (a unit of processing per hour) and bills **per second**. The price of one DBU depends on the kind of compute you pick.

| Compute type | Best for | AWS $/DBU (Premium) | Azure $/DBU (Premium, East US) | Machines billed separately? |
|---|---|---|---|---|
| Jobs Compute | Scheduled production work, such as a nightly refresh | about $0.15 | about $0.30 | Yes |
| All-Purpose Compute | Exploring and developing in notebooks | about $0.55 | about $0.55 | Yes |
| SQL Warehouse (classic) | Steady, all-day reporting | about $0.22 | about $0.22 | Yes |
| Serverless SQL | Dashboards and bursty BI use | about $0.70 to $0.75 | about $0.70 | No, included |
| Serverless Jobs | Scheduled work with no machine setup | about $0.35 | about $0.45 to $0.47 | No, included |

**How to read this**
* **Two meters or one.** On classic compute (Jobs, All-Purpose, classic SQL) your cloud provider charges separately for the virtual machines. Serverless includes the machines in its rate.
* **The biggest saving.** Moving scheduled work from All-Purpose to Jobs typically cuts the Databricks charge for that work by about three to four times, with no change to the work itself.
* **Idle time.** Serverless SQL warehouses switch off when idle, so you do not pay for a machine sitting empty overnight.
* **Not a quote.** These are approximate USD list prices from public price pages and third-party summaries (November 2026). They exclude taxes and any negotiated discount. Check the Databricks pricing calculator for your cloud and region before quoting a number.

#### Live list prices from your workspace
This reads Databricks' own price table, `system.billing.list_prices`, so you can compare the real numbers for your cloud. It lists the current price per DBU for the compute types above (all tiers and regions your account can see). If the query fails, ask an account admin to enable the `billing` system schema, or use the table above.

In [0]:
SELECT
  cloud,
  sku_name,
  currency_code,
  pricing.effective_list.default AS list_price_per_dbu
FROM system.billing.list_prices
WHERE price_end_time IS NULL
  AND usage_unit = 'DBU'
  AND (
        sku_name LIKE '%ALL_PURPOSE_COMPUTE%'
     OR sku_name LIKE '%JOBS_COMPUTE%'
     OR sku_name LIKE '%JOBS_SERVERLESS_COMPUTE%'
     OR sku_name LIKE '%SQL_COMPUTE%'
  )
  -- AND cloud = 'AWS'   -- uncomment to focus on one cloud
ORDER BY cloud, sku_name
LIMIT 50;

cloud,sku_name,currency_code,list_price_per_dbu
AWS,ENTERPRISE_ALL_PURPOSE_COMPUTE,USD,0.650000000000000000
AWS,ENTERPRISE_ALL_PURPOSE_COMPUTE_(DLT),USD,0.650000000000000000
AWS,ENTERPRISE_ALL_PURPOSE_COMPUTE_(PHOTON),USD,0.650000000000000000
AWS,ENTERPRISE_JOBS_COMPUTE,USD,0.200000000000000000
AWS,ENTERPRISE_JOBS_COMPUTE_(PHOTON),USD,0.200000000000000000
AWS,ENTERPRISE_JOBS_SERVERLESS_COMPUTE_AP_JAKARTA,USD,0.520000000000000000
AWS,ENTERPRISE_JOBS_SERVERLESS_COMPUTE_AP_MUMBAI,USD,0.470000000000000000
AWS,ENTERPRISE_JOBS_SERVERLESS_COMPUTE_AP_SEOUL,USD,0.500000000000000000
AWS,ENTERPRISE_JOBS_SERVERLESS_COMPUTE_AP_SINGAPORE,USD,0.500000000000000000
AWS,ENTERPRISE_JOBS_SERVERLESS_COMPUTE_AP_SYDNEY,USD,0.500000000000000000


## Cleanup (optional)

In [0]:
-- DROP SCHEMA IF EXISTS demo.sales CASCADE;
-- DROP CATALOG IF EXISTS demo CASCADE;